In [ ]:
import yfinance as yf
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))


In [2]:
from src.data.ingest import build_universe

tickers = build_universe()
print(len(tickers))
print(tickers[:5])
print(tickers[-4:])

102
['III.L', 'ADM.L', 'AAF.L', 'ALW.L', 'AAL.L']
['WEIR.L', 'WTB.L', '^FTSE', '^VIX']


In [3]:
from src.data.ingest import build_universe, download_prices
from pathlib import Path

tickers = build_universe()
cache = Path("../data/raw/prices.parquet")
df = download_prices(tickers, cache)

print(df.shape)
print(df.columns.get_level_values(0).unique())  # should show Close, Volume
print(df.index[[0, -1]])                         # first and last dates


Loading cached data from ../data/raw/prices.parquet
(2919, 204)
Index(['Close', 'Volume'], dtype='object', name='Price')
DatetimeIndex(['2015-01-02', '2026-05-08'], dtype='datetime64[ns]', name='Date', freq=None)


In [4]:
from src.data.ingest import clean_prices

clean_df, report = clean_prices(df)

print(clean_df.shape)
print("\nTickers dropped:")
print(report[report["drop_reason"] != ""][["non_nan_rows", "pct_missing", "drop_reason"]])
print("\nExtreme return flags (top 10):")
print(report["extreme_returns"].sort_values(ascending=False).head(10))


Dropping 7 tickers: ['AAF.L', 'CCEP.L', 'EDV.L', 'HLN.L', 'MNG.L', 'MTLN.L', 'PSH.L']
(2919, 190)

Tickers dropped:
        non_nan_rows  pct_missing                 drop_reason
Ticker                                                       
AAF.L           1733        40.63  excessive missingness >20%
CCEP.L          1795        38.51  excessive missingness >20%
EDV.L           2089        28.43  excessive missingness >20%
HLN.L            962        67.04        insufficient history
MNG.L           1653        43.37  excessive missingness >20%
MTLN.L           193        93.39        insufficient history
PSH.L           2313        20.76  excessive missingness >20%

Extreme return flags (top 10):
Ticker
^VIX      5
AAF.L     0
ABF.L     0
ADM.L     0
ALW.L     0
AAL.L     0
AUTO.L    0
AV.L      0
AZN.L     0
BA.L      0
Name: extreme_returns, dtype: int64


In [5]:
clean_df, report = clean_prices(df)
report.to_csv("../data/processed/quality_report.csv")


Dropping 7 tickers: ['AAF.L', 'CCEP.L', 'EDV.L', 'HLN.L', 'MNG.L', 'MTLN.L', 'PSH.L']
